In [1]:
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_kernels

In [2]:
data = pd.read_csv("all_participants_data.csv", index_col="Participant")

indexes = data.index
participants = set(indexes)

targetValues = data["median_valence"]
data = data.drop(columns=['median_arousal', 'median_valence'])

scaler = StandardScaler()
scaledArray = scaler.fit_transform(data)
scaledData = pd.DataFrame(scaledArray, index=indexes)
scaledData["target"] = targetValues


    


In [ ]:
import numpy as np
import pandas as pd

def pairwise_transform_vectorized(df, feature_cols, target_col):

    X = df[feature_cols].to_numpy()
    y = df[target_col].to_numpy()
    n = len(df)

    # All (i,j) index pairs where i < j
    i, j = np.triu_indices(n, 1)

    # Skip ties
    mask = y[i] != y[j]
    i, j = i[mask], j[mask]

    # Vectorized labels: 1 if y[i] > y[j], else 0
    labels = (y[i] > y[j]).astype(np.int8)

    # Vectorized difference features
    X_diff = X[i] - X[j]     

    # Build output DataFrame
    out = pd.DataFrame(X_diff, columns=feature_cols)
    out["label"] = labels

    return out

In [4]:
print(scaledData.columns.values.tolist())

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

In [5]:
featureList = scaledData.columns.values.tolist()
featureList.remove("target")

pairwiseTransformations = []


for participant in participants:
    print(participant)
    filteredData = scaledData.loc[participant]

    pairwiseTransformations.append(pairwise_transform_vectorized(filteredData, featureList, "target"))




64
65
34
37
39
41
42
45
46
16
19
21
23
56
25
26
28
30


In [6]:
#Logistic regression on the pairwise transformed data

from sklearn.linear_model import LogisticRegression

model1 = LogisticRegression()

In [7]:
#Neural network classifier with L hidden layers on the pairwise transformed data

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

class PairwiseNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2):
        super().__init__()

        layers = []
        last_dim = input_dim

        for _ in range(num_layers):
            layers.append(nn.Linear(last_dim, hidden_dim))
            layers.append(nn.ReLU())
            last_dim = hidden_dim

        # output layer for binary classification
        layers.append(nn.Linear(last_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [ ]:
#LSTM classifier with K hidden layers on the pairwise transformed data


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        """
        input_size = 1 if reshaping each feature value into a single timestep
        hidden_size = LSTM hidden dimension
        num_layers = K LSTM layers
        """
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers == 1 else 0.2
        )
        
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)
    
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out_packed, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out_packed, batch_first=True)

        last = out[:, -1, :]
        return self.fc(last)

In [9]:
from scipy.stats import pearsonr
import numpy as np

def CCcoefficient(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)
    cov = np.mean((y_true - mean_true) * (y_pred - mean_pred))

    ccc = (2 * cov) / (var_true + var_pred + (mean_true - mean_pred) ** 2)
    return ccc

In [ ]:
from sklearn.linear_model import SGDClassifier

results = []
n = len(pairwiseTransformations)

for test_idx in range(n):
    print(f"\n=== LOPO fold (test participant {test_idx}) ===")

        # Test set
    test_df = pairwiseTransformations[test_idx]
    if len(test_df) == 0:
        results.append(np.nan)
        continue

        # Train set: all except test_idx 
    train_dfs = [pairwiseTransformations[i] for i in range(n) if i != test_idx]

        # 1) Incremental scaler
    scaler = StandardScaler()
    for df in train_dfs:
        X = df.drop(columns=["label"]).values
        if len(X) > 0:
            scaler.partial_fit(X)

        # 2) Incremental logistic regression
    clf = SGDClassifier(
        loss="log_loss",
        max_iter=1,
        tol=None,
        warm_start=True,
    )
    classes = np.array([0, 1])

        # 3) Train incrementally on each participant
    for df in train_dfs:
        X = df.drop(columns=["label"]).values
        y = df["label"].values
        if len(X) == 0:
            continue
        X = scaler.transform(X)
        clf.partial_fit(X, y, classes=classes)

        # 4) Evaluate
    X_test = scaler.transform(test_df.drop(columns=["label"]).values)
    y_test = test_df["label"].values
    y_pred = clf.predict_proba(X_test)[:, 1]

    r, _ = pearsonr(y_test, y_pred)
    ccc = CCcoefficient(y_test, y_pred)

    print(r, ccc)
    results.append((r, ccc))


print(results)


=== LOPO fold (test participant 0) ===
0.028048807816922663 0.027451774510872377

=== LOPO fold (test participant 1) ===
0.09987508605003523 0.09917632899302314

=== LOPO fold (test participant 2) ===
-0.1697758830370372 -0.15805870328905874

=== LOPO fold (test participant 3) ===
0.058934474048220824 0.05826137439341349

=== LOPO fold (test participant 4) ===
0.1310189359735168 0.13003398385912884

=== LOPO fold (test participant 5) ===
-0.15332846064192818 -0.1513277977714487

=== LOPO fold (test participant 6) ===
0.10254524842591008 0.10005905344863446

=== LOPO fold (test participant 7) ===
0.23410640719997114 0.22789396482600507

=== LOPO fold (test participant 8) ===
0.21776953081781292 0.2153444182190542

=== LOPO fold (test participant 9) ===
0.014651382384224837 0.013801198022995597

=== LOPO fold (test participant 10) ===
0.022667023010986224 0.022447711027524436

=== LOPO fold (test participant 11) ===
0.24496722221327932 0.23548875707382147

=== LOPO fold (test participan

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def lopo_cv_nn(
    pairwise_dfs,
    feature_cols,
    label_col="label",
    num_layers=2,
    hidden_dim=16,
    batch_size=4096,
    epochs=10,
    lr=1e-3
):
    results = []
    n = len(pairwise_dfs)

    input_dim = len(feature_cols)

    for test_idx in range(n):
        print(f"\n=== LOPO fold (test participant {test_idx}) ===")

        test_df = pairwise_dfs[test_idx]
        if len(test_df) == 0:
            results.append({"pearsonr": np.nan, "ccc": np.nan})
            continue

        # Build model
        model = PairwiseNN(input_dim, hidden_dim, num_layers).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.BCEWithLogitsLoss()

        # Training
        model.train()
        for df in [pairwise_dfs[i] for i in range(n) if i != test_idx]:

            X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
            y = torch.tensor(df[label_col].values, dtype=torch.float32).unsqueeze(1)

            dataset = TensorDataset(X, y)
            loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

            for epoch in range(epochs):
                for X_batch, y_batch in loader:
                    X_batch = X_batch.to(device)
                    y_batch = y_batch.to(device)

                    optimizer.zero_grad()
                    logits = model(X_batch)
                    loss = loss_fn(logits, y_batch)
                    loss.backward()
                    optimizer.step()

        model.eval()
        X_test = torch.tensor(test_df[feature_cols].values, dtype=torch.float32).to(device)
        y_test = test_df[label_col].values

        with torch.no_grad():
            preds = model(X_test).cpu().numpy().flatten()
            preds_prob = 1 / (1 + np.exp(-preds)) 

        r, _ = pearsonr(y_test, preds_prob)
        ccc = CCcoefficient(y_test, preds_prob)

        print(f"Fold {test_idx} → Pearson r = {r:.4f},  CCC = {ccc:.4f}")

        results.append((r, ccc))

    return results

In [ ]:
feature_cols = [c for c in pairwiseTransformations[0].columns if c != "label"]

results = lopo_cv_nn(
    pairwiseTransformations,
    feature_cols,
    num_layers=2,   
    hidden_dim=16,   
    batch_size=1024,  
    epochs=5,        
    lr=1e-3
)

print(results)


=== LOPO fold (test participant 0) ===
Fold 0 → Pearson r = 0.1412,  CCC = 0.1312

=== LOPO fold (test participant 1) ===
Fold 1 → Pearson r = 0.2186,  CCC = 0.2170

=== LOPO fold (test participant 2) ===
Fold 2 → Pearson r = -0.1447,  CCC = -0.1344

=== LOPO fold (test participant 3) ===
Fold 3 → Pearson r = -0.0396,  CCC = -0.0395

=== LOPO fold (test participant 4) ===
